In [2]:
import pandas as pd
import numpy as np
import joblib
import json

df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

kb = joblib.load(
    "../models/recommendation_model/recommendation_knowledge_base.pkl"
)

print(df.shape)

(50000, 104)


In [3]:
def get_severity(score):

    if score <= 20:
        return "Critical"

    elif score <= 40:
        return "High"

    elif score <= 60:
        return "Medium"

    elif score <= 80:
        return "Low"

    else:
        return "Very Low"

In [4]:
def estimate_boost(score):

    if score <= 20:
        return "+20%"

    elif score <= 40:
        return "+15%"

    elif score <= 60:
        return "+10%"

    elif score <= 80:
        return "+5%"

    else:
        return "+2%"

In [5]:
def generate_context_recommendation(row):

    sector = row["sector"]

    weakness = row["weakness_1"]

    score_map = {

        "Founder Strength":
            row["founder_strength_score"],

        "Funding Strength":
            row["funding_strength_score"],

        "Market Opportunity":
            row["market_opportunity_score"],

        "Growth":
            row["growth_score"]

    }

    score = score_map[weakness]

    severity = get_severity(score)

    priority = (

        "High"
        if score < 40

        else "Medium"
        if score < 70

        else "Low"
    )

    sector_templates = kb[
        "sector_templates"
    ]

    weakness_templates = kb[
        "weakness_templates"
    ]

    sector_rec = []

    if sector in sector_templates:

        sector_rec = sector_templates[
            sector
        ]["growth_focus"]

    weakness_rec = weakness_templates[
        weakness
    ]

    recommendations = (
        weakness_rec[:3]
        +
        sector_rec[:2]
    )

    roadmap = " → ".join(
        recommendations[:3]
    )

    success_boost = estimate_boost(
        score
    )

    investor_boost = estimate_boost(
        score
    )

    return (

        severity,

        priority,

        "|".join(
            recommendations
        ),

        success_boost,

        investor_boost,

        roadmap

    )

In [6]:
results = df.apply(

    generate_context_recommendation,

    axis=1

)

df["recommendation_severity_v3"] = [

    x[0]
    for x in results
]

df["recommendation_priority_v3"] = [

    x[1]
    for x in results
]

df["personalized_recommendations"] = [

    x[2]
    for x in results
]

df["estimated_success_boost"] = [

    x[3]
    for x in results
]

df["estimated_investor_boost"] = [

    x[4]
    for x in results
]

df["improvement_roadmap_v2"] = [

    x[5]
    for x in results
]

In [7]:
df[[
    "startup_name",
    "sector",
    "weakness_1",
    "recommendation_severity_v3",
    "recommendation_priority_v3",
    "personalized_recommendations",
    "estimated_success_boost",
    "improvement_roadmap_v2"
]].head(10)

,startup_name,sector,weakness_1,recommendation_severity_v3,recommendation_priority_v3,personalized_recommendations,estimated_success_boost,improvement_roadmap_v2
0,HyperAnalytics Technologies,EdTech,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
1,HealthConnect Solutions,EdTech,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
2,InsightLoop Ventures,HealthTech,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
3,BioHub Analytics,Gaming,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
4,WaveGen Private Limited,EdTech,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
5,LaunchEngine Innovations,E-Commerce,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
6,SolarGrowth Solutions,Cybersecurity,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
7,PulseSystems Solutions,Logistics,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
8,BridgeVertex Solutions,PropTech,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...
9,BlueEdge Labs,EdTech,Founder Strength,Critical,High,Hire experienced co-founder|Build advisory boa...,+20%,Hire experienced co-founder → Build advisory b...


In [8]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("Context Recommendations Added ✅")

Context Recommendations Added ✅


In [9]:
engine = {

    "engine_name":
        "Context Aware Recommendation Engine",

    "version":
        "V3"

}

joblib.dump(

    engine,

    "../models/recommendation_model/context_aware_recommendation_engine.pkl"

)

['../models/recommendation_model/context_aware_recommendation_engine.pkl']

In [10]:
metadata = {

    "engine_name":
        "Context Aware Recommendation Engine",

    "version":
        "V3",

    "outputs":[

        "personalized_recommendations",

        "estimated_success_boost",

        "estimated_investor_boost",

        "improvement_roadmap_v2"

    ]

}

with open(

    "../models/recommendation_model/context_metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )